# Destilar la voz **Alex** (Kokoro) a PiperGenera una voz Piper con el timbre de `em_alex` (masculina, español). Todo automático: cero grabación, cero transcripción.**Parámetros (fundamentados, no arbitrarios):**- **1.300 frases** — recomendación de la comunidad Piper para fine-tuning (13.000 sería desde cero).- **Corpus fonéticamente balanceado** — 1.300 frases seleccionadas de Mozilla Common Voice (dominio público) por cobertura de difonemas del español; español natural, sin nombres extranjeros. Fijo y versionado en el repo.- **1.000 epochs**, **22.050 Hz** — valores de la doc oficial de Piper para fine-tuning.**Antes de correr:** Entorno de ejecución → Cambiar tipo → **T4 GPU**.

In [ ]:
#@title 1. GPUimport torchassert torch.cuda.is_available(), "Activá T4 GPU: Entorno de ejecución -> Cambiar tipo de entorno"print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
#@title 2. Kokoro + el corpus balanceado (fijo)!pip install -q kokoro-onnx==0.5.0!wget -q -nc -O /content/kokoro-v1.0.onnx "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx"!wget -q -nc -O /content/voices-v1.0.bin "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin"!wget -q -O /content/corpus.txt "https://raw.githubusercontent.com/Lazy-Money/Loud-Web/claude/readvox-research-slumjx/colab/corpus/corpus_es_1300.txt"frases = [l.strip() for l in open("/content/corpus.txt", encoding="utf-8") if l.strip()]print(len(frases), "frases cargadas. Ejemplo:", frases[0])

In [ ]:
#@title 3. Generar el dataset con la voz Alex (~5-10 min)import wave, numpy as np, osfrom kokoro_onnx import Kokorokokoro = Kokoro("/content/kokoro-v1.0.onnx", "/content/voices-v1.0.bin")os.makedirs("/content/dataset/wavs", exist_ok=True)rows, total = [], 0.0for i, f in enumerate(frases):    try:        samples, rate = kokoro.create(f, voice="em_alex", speed=1.0, lang="es")    except Exception as e:        print("salteada:", f[:30], e); continue    idx = np.linspace(0, len(samples)-1, int(len(samples)*22050/rate))    data = np.clip(np.interp(idx, np.arange(len(samples)), samples)*32767, -32768, 32767).astype(np.int16)    seg = len(data)/22050    if not 1.0 <= seg <= 20.0: continue    n = f"f{i:05d}"    with wave.open(f"/content/dataset/wavs/{n}.wav","wb") as w:        w.setnchannels(1); w.setsampwidth(2); w.setframerate(22050); w.writeframes(data.tobytes())    rows.append(f"{n}|{f}"); total += seg    if len(rows)%100==0: print(f"  {len(rows)} frases, {total/60:.1f} min")open("/content/dataset/metadata.csv","w",encoding="utf-8").write("\n".join(rows)+"\n")print(f"DATASET: {len(rows)} frases, {total/60:.1f} min de audio Alex")

In [ ]:
#@title 4. Instalar Piper (entrenamiento)%cd /content!git clone -q https://github.com/rhasspy/piper.git%cd /content/piper/src/python!pip install -q -e .!pip install -q "pytorch-lightning~=1.9" espeak-phonemizer librosa "numpy<2"!apt-get install -yq espeak-ng > /dev/null!bash build_monotonic_align.shprint("Piper listo")

In [ ]:
#@title 5. Preprocesar + checkpoint base%cd /content/piper/src/python!python -m piper_train.preprocess --language es --input-dir /content/dataset \  --output-dir /content/train_out --dataset-format ljspeech --single-speaker --sample-rate 22050!wget -q -nc -O /content/base.ckpt "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/es/es_ES/davefx/medium/epoch%3D2218-step%3D562840.ckpt"print("Listo para entrenar")

In [ ]:
#@title 6. Entrenar 1000 epochs (2-4 h; podés cortar y pasar a la celda 7)%cd /content/piper/src/python!python -m piper_train --dataset-dir /content/train_out --accelerator gpu --devices 1 \  --batch-size 16 --validation-split 0.0 --num-test-examples 0 \  --max_epochs 3219 --resume_from_checkpoint /content/base.ckpt \  --checkpoint-epochs 5 --precision 32 --quality medium

In [ ]:
#@title 7. Exportar y descargar la vozimport glob, shutil, osck = sorted(glob.glob("/content/train_out/lightning_logs/*/checkpoints/*.ckpt"), key=os.path.getmtime)assert ck, "Faltan checkpoints: corré la celda 6 unos epochs primero"%cd /content/piper/src/python!python -m piper_train.export_onnx "{ck[-1]}" /content/es_alex_kokoro.onnxshutil.copy("/content/train_out/config.json", "/content/es_alex_kokoro.onnx.json")from google.colab import filesfiles.download("/content/es_alex_kokoro.onnx")files.download("/content/es_alex_kokoro.onnx.json")print("Copiá ambos a tu carpeta de voces de LoudVox y elegí 'es_alex_kokoro' en Configuración")